# Prepare Dataset

Converts the Amazon Reviews 2023 gzipped JSONL data (reviews + meta) into RecBole Atomic Files. 

In [1]:
import json
import gzip
from pathlib import Path
from typing import Any
import pandas as pd

In [2]:
# --- Config ---
DATASET_NAME = "Beauty"
TARGET_CATEGORY = "Beauty_and_Personal_Care"
SOURCE_CATEGORIES = ["Beauty_and_Personal_Care", "Clothing_Shoes_and_Jewelry"]
SAMPLE_SIZE: int | None = 1_000_000 # For each category, sample this many reviews (None = use all reviews)
DATA_DIR: str = "../data"

# Only include reviews that satisfy all of the following criteria
START_DATE: str | None = "2020-01-01"
END_DATE: str | None = "2022-12-31"
MIN_RATING: int | None = None

# Data split ratios
TRAIN_RATIO: float = 0.8
VALID_RATIO: float = 0.1

# Cold vs. warm users
WARM_USER_MIN_REVIEWS: int = 5

# Sequential recommendation config
MAX_ITEM_LIST_LENGTH: int = 50

## Common utils

In [3]:
def stream_jsonl(path: str, fields: list[str] | None = None):
    with gzip.open(path, "rt", encoding="utf-8") as f:
        for _, line in enumerate(f):
            obj = json.loads(line)
            if fields is not None:
                obj = {k: obj.get(k) for k in fields}
            yield obj

def _date_to_ms(date_str: str | None) -> int | None:
    if date_str is None:
        return None
    return int(pd.Timestamp(date_str, tz="UTC").timestamp() * 1000)

def load_reviews(
    categories: list[str], sample_size: int | None = None, 
    start_date: str | None = None, end_date: str | None = None, 
    min_rating: int | None = None
) -> list[dict[str, Any]]:
    start_ts = _date_to_ms(start_date)
    end_ts = _date_to_ms(end_date)
    print(f"Filtering reviews with criteria: start_date={start_date}, end_date={end_date}, min_rating={min_rating}")

    reviews: list[dict[str, Any]] = []
    for cat in categories:
        cat_reviews: list[dict[str, Any]] = []

        path = f"{DATA_DIR}/{cat}.jsonl.gz"
        print(f"Loading reviews: {path}")
        for obj in stream_jsonl(path, fields=[
            'user_id', 'parent_asin', 'rating', 'timestamp'
        ]):
            ts: Any = obj.get("timestamp")
            rating: Any = obj.get("rating")
            if start_ts is not None and (ts is not None and ts < start_ts):
                continue
            if end_ts is not None and (ts is not None and ts > end_ts):
                continue
            if min_rating is not None and (rating is not None and rating < min_rating):
                continue
            obj["category"] = cat
            cat_reviews.append(obj)

            if sample_size is not None and len(cat_reviews) >= sample_size:
                print(f"Reached sample size limit ({sample_size} reviews). Stopping.")
                break

        reviews.extend(cat_reviews)
    return reviews


## Load reviews

In [4]:
reviews = load_reviews(
    SOURCE_CATEGORIES, sample_size=SAMPLE_SIZE, start_date=START_DATE, end_date=END_DATE,
    min_rating=MIN_RATING
)
print(f"Loaded {len(reviews)} reviews")

Filtering reviews with criteria: start_date=2020-01-01, end_date=2022-12-31, min_rating=None
Loading reviews: ../data/Beauty_and_Personal_Care.jsonl.gz
Reached sample size limit (1000000 reviews). Stopping.
Loading reviews: ../data/Clothing_Shoes_and_Jewelry.jsonl.gz
Reached sample size limit (1000000 reviews). Stopping.
Loaded 2000000 reviews


In [5]:
df_reviews = pd.DataFrame(reviews)
display(df_reviews.sample(10))
df_reviews.info()

,user_id,parent_asin,rating,timestamp,category
1199927,AHOX3MRFYM6UOD7SXHUK5OEGNPSQ,B07XZ6RWF3,5.0,1655171084409,Clothing_Shoes_and_Jewelry
836490,AHYZDKHNSONUFM7V36ZYN6B745WQ,B09H42LJDT,5.0,1667230911157,Beauty_and_Personal_Care
1116965,AHEZLWQWB6LAQ332J552XN7KRATQ,B0B8X3MQ2Q,5.0,1669345941377,Clothing_Shoes_and_Jewelry
1833460,AFI6XCKPUVT72PSP7NGHYFXFOYUQ,B07CPG4X9X,5.0,1578080693641,Clothing_Shoes_and_Jewelry
1650273,AGU2S5TT7X4VT5FCJBWJYOPUKORA,B07HB39WBN,3.0,1614976419658,Clothing_Shoes_and_Jewelry
891507,AFMLHELGD4WJSACKP7LJ4UJ5AKUA,B0C57HN4WB,5.0,1662587611747,Beauty_and_Personal_Care
986176,AG4CQWF5SDEGB7HZXMQYUSEWO5ZQ,B09WYMYMDW,5.0,1670372461675,Beauty_and_Personal_Care
1560066,AEYN3SGEY2JWP7EZDC52Y5GNVOYA,B07SQDHHSW,5.0,1591288058971,Clothing_Shoes_and_Jewelry
1633954,AE6QUUOKYBGENJY7XMUX57DE5RXQ,B08P3YBW2V,5.0,1672056584706,Clothing_Shoes_and_Jewelry
362630,AFT5CDS53T6DBNDJDQ7VVXE4YFLA,B00A1P02YS,5.0,1593379995234,Beauty_and_Personal_Care


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000000 entries, 0 to 1999999
Data columns (total 5 columns):
 #   Column       Dtype  
---  ------       -----  
 0   user_id      object 
 1   parent_asin  object 
 2   rating       float64
 3   timestamp    int64  
 4   category     object 
dtypes: float64(1), int64(1), object(3)
memory usage: 76.3+ MB


## Map user/item IDs to integers

In [6]:
# As RecBole expects integer IDs, we need to create mappings from the original string IDs to integers.
user_ids: set[str] = set()
item_ids: set[str] = set()
for r in reviews:
    user_ids.add(r["user_id"])
    item_ids.add(r["parent_asin"])

user_map: dict[str, int] = {uid: i+1 for i, uid in enumerate(sorted(user_ids))}
item_map: dict[str, int] = {pid: i+1 for i, pid in enumerate(sorted(item_ids))}
print(f"Users: {len(user_map):,}  Items: {len(item_map):,}")

Users: 445,709  Items: 587,481


In [7]:
df_reviews['uid'] = df_reviews['user_id'].map(user_map)
df_reviews['iid'] = df_reviews['parent_asin'].map(item_map)

## Split train/valid/test with cutoff timestamps

In [8]:
# Only reviews from the target category
df_target_reviews = df_reviews[df_reviews["category"] == TARGET_CATEGORY]
df_target_reviews

,user_id,parent_asin,rating,timestamp,category,uid,iid
0,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B00Z03RC80,1.0,1616743454733,Beauty_and_Personal_Care,170879,42983
1,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B085PRT2MP,1.0,1614915977684,Beauty_and_Personal_Care,170879,282873
2,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B08G81QQ9L,5.0,1612052493701,Beauty_and_Personal_Care,170879,335445
3,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B07YYG76X1,1.0,1609700981786,Beauty_and_Personal_Care,170879,245283
4,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B07X4FKLNK,3.0,1581313195358,Beauty_and_Personal_Care,170879,231310
...,...,...,...,...,...,...,...
999995,AGEXNM33KWH22HOZLUD6AIIOF6JA,B008IEUERE,5.0,1647731000766,Beauty_and_Personal_Care,261103,18176
999996,AGEXNM33KWH22HOZLUD6AIIOF6JA,B08ZV6QWMC,2.0,1647730900805,Beauty_and_Personal_Care,261103,402899
999997,AGEXNM33KWH22HOZLUD6AIIOF6JA,B077ZCNZYP,5.0,1647730735686,Beauty_and_Personal_Care,261103,104800
999998,AGEXNM33KWH22HOZLUD6AIIOF6JA,B0BDW22ZVC,5.0,1647730537972,Beauty_and_Personal_Care,261103,543907


In [9]:
# Determine cutoff timestamps for train/valid/test splits based on the target category only
timestamps = sorted(df_target_reviews["timestamp"].values)
train_end_ts = timestamps[int(len(timestamps) * TRAIN_RATIO)]
valid_end_ts = timestamps[int(len(timestamps) * (TRAIN_RATIO + VALID_RATIO))]
print(f"Train end timestamp: {train_end_ts} ({pd.Timestamp(train_end_ts, unit='ms', tz='UTC')})")
print(f"Valid end timestamp: {valid_end_ts} ({pd.Timestamp(valid_end_ts, unit='ms', tz='UTC')})")

Train end timestamp: 1654364240346 (2022-06-04 17:37:20.346000+00:00)
Valid end timestamp: 1663518944685 (2022-09-18 16:35:44.685000+00:00)


In [10]:
df_reviews_train = df_reviews[df_reviews["timestamp"] <= train_end_ts]
df_reviews_valid = df_reviews[(df_reviews["timestamp"] > train_end_ts) & (df_reviews["timestamp"] <= valid_end_ts)]
df_reviews_test = df_reviews[df_reviews["timestamp"] > valid_end_ts]
print(f"Train reviews: {len(df_reviews_train)}")
print(f"Valid reviews: {len(df_reviews_valid)}")
print(f"Test reviews: {len(df_reviews_test)}")

Train reviews: 1603031
Valid reviews: 195409
Test reviews: 201560


In [11]:
df_target_reviews_train = df_target_reviews[df_target_reviews["timestamp"] <= train_end_ts]
df_target_reviews_valid = df_target_reviews[(df_target_reviews["timestamp"] > train_end_ts) & (df_target_reviews["timestamp"] <= valid_end_ts)]
df_target_reviews_test = df_target_reviews[df_target_reviews["timestamp"] > valid_end_ts]
print(f"Target category train reviews: {len(df_target_reviews_train)}")
print(f"Target category valid reviews: {len(df_target_reviews_valid)}")
print(f"Target category test reviews: {len(df_target_reviews_test)}")

Target category train reviews: 800001
Target category valid reviews: 100000
Target category test reviews: 99999


## Define cold vs. warm users with train data in target category

In [12]:
train_counts = df_target_reviews_train.groupby("uid", sort=False).size().rename("num_train")
valid_counts = df_target_reviews_valid.groupby("uid", sort=False).size().rename("num_valid")
test_counts = df_target_reviews_test.groupby("uid", sort=False).size().rename("num_test")

df_user_target_counts = (
    pd.concat([train_counts, valid_counts, test_counts], axis=1)
    .fillna(0)
    .astype(int)
    .reset_index()
)

df_train_users = df_user_target_counts[df_user_target_counts["num_train"] > 0]
cold_user_ids: set[str] = set(df_train_users[df_train_users["num_train"] < WARM_USER_MIN_REVIEWS]["uid"])
warm_user_ids: set[str] = set(df_train_users[df_train_users["num_train"] >= WARM_USER_MIN_REVIEWS]["uid"])
print(f"Warm users (>= {WARM_USER_MIN_REVIEWS} reviews): {len(warm_user_ids)}")
print(f"Cold users (< {WARM_USER_MIN_REVIEWS} reviews): {len(cold_user_ids)}")

Warm users (>= 5 reviews): 36001
Cold users (< 5 reviews): 256076


## Define item data

In [13]:
df_items = (
    df_reviews[["iid", "category"]]
    .assign(is_target=(df_reviews["category"] == TARGET_CATEGORY))
    .groupby("iid", as_index=False)["is_target"]
    .any()
)
target_item_ids: set[str] = set(df_items[df_items["is_target"]]["iid"])
df_items

,iid,is_target
0,1,True
1,2,False
2,3,True
3,4,False
4,5,False
...,...,...
587476,587477,False
587477,587478,True
587478,587479,True
587479,587480,True


## Generate user-item history for sequential recommendation

In [14]:
df_sorted: pd.DataFrame = pd.concat(
    [
        df_reviews_train.assign(split="train"),
        df_reviews_valid.assign(split="valid"),
        df_reviews_test.assign(split="test"),
    ],
    ignore_index=True,
).sort_values(
    ["uid", "timestamp"],
    ascending=[True, True],
    kind="mergesort",
)

def build_history(series: pd.Series) -> pd.Series:
    result: list[str] = []
    history: list[int] = []
    for val in series:
        result.append(" ".join(map(str, history[-MAX_ITEM_LIST_LENGTH:])))
        history.append(val)
    return pd.Series(result, index=series.index, dtype='str')

# History for all items (not just target category)
df_sorted["iid_list"] = (
    df_sorted.groupby("uid", sort=False)["iid"]
    .transform(build_history)
)

# History for items in the target category only
df_sorted["target_iid_list"] = ""
target_mask = df_sorted['iid'].isin(target_item_ids)
df_sorted.loc[target_mask, "target_iid_list"] = (
    df_sorted[target_mask]
    .groupby("uid", sort=False)["iid"]
    .transform(build_history)
)

df_sorted

,user_id,parent_asin,rating,timestamp,category,uid,iid,split,iid_list,target_iid_list
97938,AE222ABE7SFNVT34H5XDXASHAP5A,B0BCSLPSTK,5.0,1617223937658,Beauty_and_Personal_Care,1,541959,train,,
623854,AE2235Q53V246ISFXCSESYHAVNUA,B0759HSBQC,2.0,1605156655266,Beauty_and_Personal_Care,2,94373,train,,
623853,AE2235Q53V246ISFXCSESYHAVNUA,B091VVDBR2,4.0,1605156954165,Beauty_and_Personal_Care,2,407321,train,94373,94373
623852,AE2235Q53V246ISFXCSESYHAVNUA,B088ZHJMHD,3.0,1605157008463,Beauty_and_Personal_Care,2,303042,train,94373 407321,94373 407321
623851,AE2235Q53V246ISFXCSESYHAVNUA,B09XS5V968,1.0,1605157165591,Beauty_and_Personal_Care,2,506019,train,94373 407321 303042,94373 407321 303042
...,...,...,...,...,...,...,...,...,...,...
924470,AHZZZZPE45DYV2WZ2MYXZRHWSEKA,B08WZ96365,5.0,1653161599224,Clothing_Shoes_and_Jewelry,445709,393019,train,155839 519196 27576 89969 550889 122645 568053...,
1717690,AHZZZZPE45DYV2WZ2MYXZRHWSEKA,B09R1MQLK7,3.0,1662166701855,Clothing_Shoes_and_Jewelry,445709,489436,valid,155839 519196 27576 89969 550889 122645 568053...,
1611920,AHZZZZPE45DYV2WZ2MYXZRHWSEKA,B0C4XYMPLS,5.0,1662166879273,Beauty_and_Personal_Care,445709,579429,valid,155839 519196 27576 89969 550889 122645 568053...,
1717689,AHZZZZPE45DYV2WZ2MYXZRHWSEKA,B094JWR1ZX,5.0,1662166953954,Clothing_Shoes_and_Jewelry,445709,418722,valid,155839 519196 27576 89969 550889 122645 568053...,


## Write atomic files

In [15]:
def get_user_category(uid: str) -> int:
    if uid in warm_user_ids:
        return 0 # Warm user
    elif uid in cold_user_ids:
        return 1 # Cold user
    else:
        return 2 # New user
    
def write_user_file(path: Path, uids: set[str]) -> None:
    row_count: int = 0

    with path.open("w", encoding="utf-8") as f:
        f.write(
            "user_id:token\t"
            "category:token\n"
        )

        for uid in uids:
            f.write(
                f"{uid}\t"
                f"{get_user_category(uid)}\n"
            )
            row_count += 1

    print(f"Wrote {path} ({row_count:,} rows)")

def write_item_file(path: Path, df: pd.DataFrame) -> None:
    row_count: int = 0

    with path.open("w", encoding="utf-8") as f:
        f.write(
            "item_id:token\t"
            "is_target:float\n"
        )

        for _, row in df.iterrows():
            f.write(
                f"{row['iid']}\t"
                f"{int(row['is_target'])}\n"
            )
            row_count += 1

    print(f"Wrote {path} ({row_count:,} rows)")

def write_inter_file(
    path: Path, df: pd.DataFrame,
    item_id_list_field: str
) -> None:
    row_count: int = 0

    with path.open("w", encoding="utf-8") as f:
        f.write(
            "user_id:token\t"
            "item_id:token\t"
            "rating:float\t"
            "timestamp:float\t"
            "item_id_list:token_seq\n"
        )

        for _, row in df.iterrows():
            f.write(
                f"{row['uid']}\t"
                f"{row['iid']}\t"
                f"{row['rating']}\t"
                f"{row['timestamp']}\t"
                f"{row[item_id_list_field]}\n"
            )
            row_count += 1

    print(f"Wrote {path} ({row_count:,} rows)")


In [16]:
df_target = df_sorted[df_sorted['iid'].isin(target_item_ids)]
df_target_train = df_target[df_target['split'] == 'train']
df_target_valid = df_target[df_target['split'] == 'valid']
df_target_test = df_target[df_target['split'] == 'test']

### Target category

In [17]:
# History only for items in the target category
target_dataset_prefix = Path(DATA_DIR) / "target" / "target"
target_dataset_prefix.parent.mkdir(parents=True, exist_ok=True)
write_inter_file(target_dataset_prefix.with_suffix(".train.inter"), 
                 df_target_train, item_id_list_field="target_iid_list")
write_inter_file(target_dataset_prefix.with_suffix(".valid.inter"), 
                 df_target_valid, item_id_list_field="target_iid_list")
write_inter_file(target_dataset_prefix.with_suffix(".test.inter"), 
                 df_target_test, item_id_list_field="target_iid_list")
write_user_file(target_dataset_prefix.with_suffix(".user"), df_target['uid'].unique())
write_item_file(target_dataset_prefix.with_suffix(".item"), df_items[df_items["is_target"]])

Wrote ../data/target/target.train.inter (800,001 rows)
Wrote ../data/target/target.valid.inter (100,000 rows)
Wrote ../data/target/target.test.inter (99,999 rows)
Wrote ../data/target/target.user (330,737 rows)
Wrote ../data/target/target.item (209,389 rows)


### Cross category 

In [18]:
# History includes all cross-category items (not just target category)
cross_dataset_prefix = Path(DATA_DIR) / "cross" / "cross"
cross_dataset_prefix.parent.mkdir(parents=True, exist_ok=True)
write_inter_file(cross_dataset_prefix.with_suffix(".train.inter"), 
                 df_target_train, item_id_list_field="iid_list")
write_inter_file(cross_dataset_prefix.with_suffix(".valid.inter"), 
                 df_target_valid, item_id_list_field="iid_list")
write_inter_file(cross_dataset_prefix.with_suffix(".test.inter"), 
                 df_target_test, item_id_list_field="iid_list")
write_user_file(cross_dataset_prefix.with_suffix(".user"), df_target['uid'].unique())
write_item_file(cross_dataset_prefix.with_suffix(".item"), df_items)

Wrote ../data/cross/cross.train.inter (800,001 rows)
Wrote ../data/cross/cross.valid.inter (100,000 rows)
Wrote ../data/cross/cross.test.inter (99,999 rows)
Wrote ../data/cross/cross.user (330,737 rows)
Wrote ../data/cross/cross.item (587,481 rows)
